In [55]:
import pandas as pd
from pathlib import Path
import csv

ordner = Path(r"C:\Users\golde\PycharmProjects\pred_main_project_brueggen\data\lstm ready data")
stoer = pd.read_csv(
    ordner / "aufschreibung_mta_clean_gesamt_v10_mit_fehlenden_zeitfenstern.csv",
    sep=",",
    engine="python"
)

prozess = pd.read_csv(
    ordner / "M200310_processdata_bereinigt_2024_206_gesamt.csv",
    sep=";",
    engine="python"
)
print("Störliste Spalten:")
print(stoer.columns)

print("\nProzessliste Spalten:")
print(prozess.columns)

Störliste Spalten:
Index(['Datum', 'Wochentag', 'KW', 'DatumNEU', 'Jahr', 'Monat', 'Tag',
       'Quartal', 'Schicht', 'Zeit von', 'Zeit bis', 'Dauer Arbeits-zeit',
       'Anzahl MA', 'Menge N.i. O.', 'Menge i. O. L4', 'Menge i. O. L5',
       'MengeGesamt', 'Dauer Org-Mangel', 'Dauer Anlagen-Ausfall',
       'Störung aufgrund Vormaterial', 'Dauer Anlagen-Ausfall intern',
       'Dauer Logistik- Defizite', 'Station/ OP', 'Bemerkung', 'Anzahl/ Std.',
       'Sollzeit/ Stück (Min)', 'Zeit_von_min', 'Zeit_bis_min',
       'Station/ OP_raw', 'Station/ OP_1', 'Station/ OP_2', 'Fehlercode',
       'Bemerkung_norm', 'Bemerkung_std', 'Störfall',
       'Anzahl Störfälle Zeitfenster'],
      dtype='object')

Prozessliste Spalten:
Index(['Kundenauftragsnummer', 'Kundenauftragspos.', 'Serialnummer', 'Auftrag',
       'Materialnummer', 'Material-Text', 'Vorgang', 'Arbeitsplatz',
       'Kurztext Vorgang', 'Vorgangsmenge (MEINH)',
       'Rückgem. Gutmenge (GMEIN)', 'Basismengeneinheit (=GMEIN)',


In [56]:
stoer["Datum"] = pd.to_datetime(
    stoer["Datum"],
    errors="coerce",
    dayfirst=True
)

prozess["Ende Durchf.(Dat.)"] = pd.to_datetime(
    prozess["Ende Durchf.(Dat.)"],
    errors="coerce",
    dayfirst=True
)
stoer["Zeit von"] = pd.to_datetime(stoer["Zeit von"], errors="coerce").dt.strftime("%H:%M:%S")
stoer["Zeit bis"] = pd.to_datetime(stoer["Zeit bis"], errors="coerce").dt.strftime("%H:%M:%S")

prozess["zeit_von"] = pd.to_datetime(prozess["zeit_von"], errors="coerce").dt.strftime("%H:%M:%S")
prozess["zeit_bis"] = pd.to_datetime(prozess["zeit_bis"], errors="coerce").dt.strftime("%H:%M:%S")
prozess_agg = (
    prozess.groupby(
        ["Ende Durchf.(Dat.)", "zeit_von", "zeit_bis"],
        as_index=False
    )
    .agg({
        "Kundenauftragsnummer": lambda x: "|".join(
            pd.unique(x.dropna().astype(str).str.strip())
        ),
        "Materialnummer": lambda x: "|".join(
            pd.unique(x.dropna().astype(str).str.strip())
        )
    })
)
ergebnis = stoer.merge(
    prozess_agg,
    left_on=["Datum", "Zeit von", "Zeit bis"],
    right_on=["Ende Durchf.(Dat.)", "zeit_von", "zeit_bis"],
    how="left"
)
ergebnis.drop(
    columns=["Ende Durchf.(Dat.)", "zeit_von", "zeit_bis"],
    inplace=True
)
ergebnis["Kundenauftragsnummer"] = ergebnis["Kundenauftragsnummer"].fillna("")
ergebnis["Materialnummer"] = ergebnis["Materialnummer"].fillna("")
ergebnis.head()

C:\Users\golde\AppData\Local\Temp\ipykernel_36040\2830807004.py:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  stoer["Zeit von"] = pd.to_datetime(stoer["Zeit von"], errors="coerce").dt.strftime("%H:%M:%S")
C:\Users\golde\AppData\Local\Temp\ipykernel_36040\2830807004.py:13: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  stoer["Zeit bis"] = pd.to_datetime(stoer["Zeit bis"], errors="coerce").dt.strftime("%H:%M:%S")
C:\Users\golde\AppData\Local\Temp\ipykernel_36040\2830807004.py:15: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  prozess["zeit_von"] = pd.to_datetime(prozess["ze

,Datum,Wochentag,KW,DatumNEU,Jahr,Monat,Tag,Quartal,Schicht,Zeit von,...,Station/ OP_raw,Station/ OP_1,Station/ OP_2,Fehlercode,Bemerkung_norm,Bemerkung_std,Störfall,Anzahl Störfälle Zeitfenster,Kundenauftragsnummer,Materialnummer
0,2024-08-01,1,2024/02,2024-01-08,2024,1,8,1,f,04:45:00,...,NaN,NaN,NaN,NaN,wartungsplan,wartungsplan,J,2,30311039.0,515102020.0
1,2024-08-01,1,2024/02,2024-01-08,2024,1,8,1,f,04:45:00,...,R 05,R 05,NaN,NaN,vakuum probleme. robtech informiert,vakuum probleme. robtech informiert,J,2,30311039.0,515102020.0
2,2024-08-01,1,2024/02,2024-01-08,2024,1,8,1,f,06:00:00,...,R 05,R 05,NaN,NaN,vakuum probleme. robtech informiert,vakuum probleme. robtech informiert,J,1,30311039.0,515102020.0
3,2024-08-01,1,2024/02,2024-01-08,2024,1,8,1,f,07:00:00,...,NaN,NaN,NaN,NaN,NaN,NaN,N,0,30311039.0,515102020.0
4,2024-08-01,1,2024/02,2024-01-08,2024,1,8,1,f,08:00:00,...,NaN,NaN,NaN,NaN,NaN,NaN,N,0,30311039.0,515102020.0


In [70]:
df = ergebnis.copy()

# echte Zeitachse (WICHTIG für alle Zeitfeatures)
df["timestamp"] = pd.to_datetime(
    df["DatumNEU"].astype(str) + " " + df["Zeit von"].astype(str),
    errors="coerce"
)
df["timestamp"].head(10)

0   2024-01-08 04:45:00
1   2024-01-08 04:45:00
2   2024-01-08 06:00:00
3   2024-01-08 07:00:00
4   2024-01-08 08:00:00
5   2024-01-08 09:30:00
6   2024-01-08 10:00:00
7   2024-01-08 11:00:00
8   2024-01-08 12:00:00
9   2024-01-08 13:15:00
Name: timestamp, dtype: datetime64[ns]

In [73]:

df = df.sort_values(by="timestamp")

# Wochenende
df["Wochenende"] = df["timestamp"].dt.weekday.apply(
    lambda x: "J" if x >= 5 else "N"
)

# Störfall bin
df["Stoerfall_bin"] = df["Störfall"].map({"J": 1, "N": 0}).fillna(0)

# letzter Störfall (korrekt auf timestamp Basis)
df["last_fail_time"] = df["timestamp"].where(df["Stoerfall_bin"] == 1)
df["last_fail_time"] = df["last_fail_time"].ffill()

# Zeit seit letztem Fehler
df["Zeit_seit_letztem_Fehler_min"] = (
    (df["timestamp"] - df["last_fail_time"]).dt.total_seconds() / 60
)

# Rolling Features auf Zeitindex
#df = df.dropna(subset=["timestamp"])
df = df.set_index("timestamp")

df["Stoerfall_7d_mean"] = df["Stoerfall_bin"].rolling("7D").mean()
df["Stoerfall_30d_mean"] = df["Stoerfall_bin"].rolling("30D").mean()

df = df.reset_index()

# Analyse: Zeit bis nächster Ausfall
stoer_times = df[df["Stoerfall_bin"] == 1]["timestamp"]

time_diff = stoer_times.diff().dt.total_seconds() / 60
avg_time_to_failure = time_diff.mean()

print("Ø Zeit bis nächster Ausfall (Min):", avg_time_to_failure)

df["Avg_Time_To_Failure_min"] = avg_time_to_failure

df.head()

Ø Zeit bis nächster Ausfall (Min): 249.2826617826618


,timestamp,DatumNEU,Datum,Wochentag,KW,Jahr,Monat,Tag,Quartal,Schicht,...,Kundenauftragsnummer,Materialnummer,Wochenende,letzter_stoerfall_time,Zeit_seit_letztem_Fehler_min,Stoerfall_7d_mean,Stoerfall_30d_mean,Avg_Time_To_Failure_min,last_fail_time,Stoerfall_bin
0,2024-01-08 04:45:00,2024-01-08,2024-08-01,1,2024/02,2024,1,8,1,f,...,30311039.0,515102020.0,N,2024-01-08,0.0,1.00,1.00,249.282662,2024-01-08 04:45:00,1
1,2024-01-08 04:45:00,2024-01-08,2024-08-01,1,2024/02,2024,1,8,1,f,...,30311039.0,515102020.0,N,2024-01-08,0.0,1.00,1.00,249.282662,2024-01-08 04:45:00,1
2,2024-01-08 06:00:00,2024-01-08,2024-08-01,1,2024/02,2024,1,8,1,f,...,30311039.0,515102020.0,N,2024-01-08,0.0,1.00,1.00,249.282662,2024-01-08 06:00:00,1
3,2024-01-08 07:00:00,2024-01-08,2024-08-01,1,2024/02,2024,1,8,1,f,...,30311039.0,515102020.0,N,2024-01-08,60.0,0.75,0.75,249.282662,2024-01-08 06:00:00,0
4,2024-01-08 08:00:00,2024-01-08,2024-08-01,1,2024/02,2024,1,8,1,f,...,30311039.0,515102020.0,N,2024-01-08,120.0,0.60,0.60,249.282662,2024-01-08 06:00:00,0


In [75]:
df = df.drop(columns=["Stoerfall_bin", "timestamp", "letzter_stoerfall_time"], errors="ignore")
ergebnis = df
ergebnis.to_csv(
    ordner / "stoerliste_mit_auftragsdaten_2024_2026.csv",
    sep=";",
    index=False,
    encoding="utf-8-sig",
    quoting=csv.QUOTE_NONNUMERIC
)
print("Anzahl Zeilen Störliste:", len(stoer))
print("Anzahl Zeilen Ergebnis:", len(ergebnis))
df.head()

Anzahl Zeilen Störliste: 8983
Anzahl Zeilen Ergebnis: 8980


,DatumNEU,Datum,Wochentag,KW,Jahr,Monat,Tag,Quartal,Schicht,Zeit von,...,Störfall,Anzahl Störfälle Zeitfenster,Kundenauftragsnummer,Materialnummer,Wochenende,Zeit_seit_letztem_Fehler_min,Stoerfall_7d_mean,Stoerfall_30d_mean,Avg_Time_To_Failure_min,last_fail_time
0,2024-01-08,2024-08-01,1,2024/02,2024,1,8,1,f,04:45:00,...,J,2,30311039.0,515102020.0,N,0.0,1.00,1.00,249.282662,2024-01-08 04:45:00
1,2024-01-08,2024-08-01,1,2024/02,2024,1,8,1,f,04:45:00,...,J,2,30311039.0,515102020.0,N,0.0,1.00,1.00,249.282662,2024-01-08 04:45:00
2,2024-01-08,2024-08-01,1,2024/02,2024,1,8,1,f,06:00:00,...,J,1,30311039.0,515102020.0,N,0.0,1.00,1.00,249.282662,2024-01-08 06:00:00
3,2024-01-08,2024-08-01,1,2024/02,2024,1,8,1,f,07:00:00,...,N,0,30311039.0,515102020.0,N,60.0,0.75,0.75,249.282662,2024-01-08 06:00:00
4,2024-01-08,2024-08-01,1,2024/02,2024,1,8,1,f,08:00:00,...,N,0,30311039.0,515102020.0,N,120.0,0.60,0.60,249.282662,2024-01-08 06:00:00
